In [ ]:
%load_ext autoreload
%autoreload 2
import os
import numpy as np
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from src.constants import BASEDIR
from src.data.objects import ProteinDocument
import hydra
from hydra import initialize_config_dir, compose

os.chdir(BASEDIR)

# DATASET_NAME = "foldseek_representatives_with_interleaved_coords"
DATASET_NAME = "cath_43_interleaved"
MAX_TOKENS = 1026
MASK_BELOW_PLDDT = 80

TODO: figure out how to change namespace config gets loaded to (@ - package)

critical for example data.

In [ ]:
with initialize_config_dir(os.path.join(BASEDIR, "configs")):
    # , "data.dataset.num_proc=8"
    cfg = compose(config_name=f"data/dataset/{DATASET_NAME}", overrides=["data.dataset.split_name=validation"],)
    tokenizer_cfg = compose(
        config_name="tokenizer/profam",
    )

In [ ]:
dataset_builder = hydra.utils.instantiate(cfg.data.dataset, _convert_="partial")
tokenizer = hydra.utils.instantiate(tokenizer_cfg.tokenizer)

In [ ]:
dataset_builder.preprocessor.transform_fns

In [ ]:
data = dataset_builder.load(data_dir="data", world_size=1, verbose=True)

In [ ]:
len(data)

In [ ]:
%%prun
protein = data[0]
proteins = ProteinDocument.from_proteins([protein], representative_accession=protein.accession, identifier=protein.accession)
example = dataset_builder.preprocessor.preprocess_protein_data(proteins, tokenizer, max_tokens=MAX_TOKENS, return_tensors=False)

In [ ]:
feature_names = ["input_ids", "attention_mask", "labels", "seq_pos", "ds_name", "identifier", "total_num_sequences", "aa_mask", "coords", "coords_mask", "interleaved_coords_mask", "structure_mask", "plddts"]
processed_data = dataset_builder.process(data, tokenizer, max_tokens_per_example=MAX_TOKENS, feature_names=feature_names)

In [ ]:
iterator = iter(processed_data)
_ = next(iterator)
example = next(iterator)

In [ ]:
example.keys()

In [ ]:
example["coords"].shape

In [ ]:
np.asarray(example["input_ids"])[:200]

In [ ]:
np.asarray(example["coords"])[:200, 0,0]

TODO: add pymol visualisation.